# Test: `fetch_fred_yields`

Verifies that `fred.py` fetches correctly, produces the right shape and format,
and saves a valid parquet file to `data/processed/`.

In [1]:
import sys
sys.path.insert(0, "../src")

from termstructure.ingest.fred import fetch_fred_yields

## Check 1 — Fetch and inspect the DataFrame

In [2]:
df = fetch_fred_yields("2000-01-01", "2024-12-31")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nFirst 12 rows (should be 6 maturities for the first 2 dates):")
df.head(12)

  Fetching DGS3MO... done (11,644 rows)
  Fetching DGS1... done (16,774 rows)
  Fetching DGS2... done (13,014 rows)
  Fetching DGS5... done (16,774 rows)
  Fetching DGS10... done (16,774 rows)
  Fetching DGS30... done (12,829 rows)
Saved 37,518 rows to data\processed\cmt_yields.parquet
Shape: (37518, 3)

Columns: ['date', 'yield_pct', 'maturity_years']

Dtypes:
 date              datetime64[us]
yield_pct                float64
maturity_years           float64
dtype: object

First 12 rows (should be 6 maturities for the first 2 dates):


,date,yield_pct,maturity_years
0,2000-01-03,5.48,0.25
1,2000-01-03,6.09,1.00
2,2000-01-03,6.38,2.00
3,2000-01-03,6.50,5.00
4,2000-01-03,6.58,10.00
5,2000-01-03,6.61,30.00
6,2000-01-04,5.43,0.25
7,2000-01-04,6.00,1.00
8,2000-01-04,6.30,2.00
9,2000-01-04,6.40,5.00


## Check 2 — Exactly 6 maturities per date

In [3]:
rows_per_date = df.groupby("date").size()
print("Min rows per date:", rows_per_date.min())
print("Max rows per date:", rows_per_date.max())
print("Any date with != 6 rows:", (rows_per_date != 6).sum())

# Should print min=6, max=6, and 0 anomalous dates

Min rows per date: 6
Max rows per date: 6
Any date with != 6 rows: 0


## Check 3 — No NaNs, yields in a sensible range

In [4]:
print("NaNs in yield_pct:", df["yield_pct"].isna().sum())
print("\nyield_pct stats:")
print(df["yield_pct"].describe().round(3))

# Yields should be between ~0 and ~10 for this date range
assert df["yield_pct"].isna().sum() == 0, "Found unexpected NaNs"
assert df["yield_pct"].min() >= 0, "Negative yield found"
assert df["yield_pct"].max() <= 20, "Implausibly high yield found"
print("\nAll range checks passed.")

NaNs in yield_pct: 0

yield_pct stats:
count    37518.000
mean         2.660
std          1.788
min          0.000
25%          1.120
50%          2.550
75%          4.250
max          6.930
Name: yield_pct, dtype: float64

All range checks passed.


## Check 4 — Parquet file exists and round-trips cleanly

In [ ]:
import pandas as pd
from termstructure.ingest.fred import OUTPUT_PATH

print("Saving to:", OUTPUT_PATH)
assert OUTPUT_PATH.exists(), f"Parquet file not found at {OUTPUT_PATH}"

df_disk = pd.read_parquet(OUTPUT_PATH)
print("Loaded from disk — shape:", df_disk.shape)
print("In-memory df shape:      ", df.shape)
assert df_disk.shape == df.shape, "Shape mismatch between returned df and parquet"
print("\nRound-trip check passed.")

## Check 5 — Spot-check a known value

Cross-reference one value against what FRED shows on its website.
The 10Y yield on 2023-01-03 should be around 3.88%.

In [ ]:
spot = df[(df["date"] == "2023-01-03") & (df["maturity_years"] == 10.0)]
print(spot)

val = spot["yield_pct"].iloc[0]
assert abs(val - 3.79) < 0.05, f"10Y on 2023-01-03 expected ~3.79%, got {val}"
print(f"\nSpot check passed: 10Y on 2023-01-03 = {val:.2f}%")